# Silver Layer - AdventureWorks

Reads AdventureWorks bronze data from ADLS Gen2 (via Unity Catalog external location) and transforms it into the silver layer.

## 1. Load bronze data

Each bronze dataset sits in its own folder in the `bronze` container as a single CSV file. `read_bronze()` reads a folder with a header row and `inferSchema`, so dates and numbers come back typed (for example, `Date` is a `DateType`).

Access goes through the Unity Catalog external location **`bronze_ext`**, which uses the `adventureworks_cred` storage credential (Azure access connector / managed identity). No storage keys or secrets are needed in the notebook.

In [0]:
# Access via Unity Catalog external location `bronze_ext` (managed identity) - no keys/secrets needed
bronze = "abfss://bronze@advworksstorageacc.dfs.core.windows.net"

def read_bronze(folder):
    return (spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", True)
            .load(f"{bronze}/{folder}"))

df_calendar     = read_bronze("Calendar")
df_customers    = read_bronze("Customers")
df_prod_cat     = read_bronze("Product_Categories")
df_prod_subcat  = read_bronze("Product_Subcategories")
df_products     = read_bronze("Products")
df_returns      = read_bronze("Returns")
df_sales_2015   = read_bronze("Sales_2015")
df_sales_2016   = read_bronze("Sales_2016")
df_sales_2017   = read_bronze("Sales_2017")
df_territories  = read_bronze("Territories")

In [0]:
dfs = {
    "calendar": df_calendar, "customers": df_customers, "product_categories": df_prod_cat,
    "product_subcategories": df_prod_subcat, "products": df_products, "returns": df_returns,
    "sales_2015": df_sales_2015, "sales_2016": df_sales_2016, "sales_2017": df_sales_2017,
    "territories": df_territories,
}
summary = [(name, df.count(), len(df.columns), ", ".join(df.columns)) for name, df in dfs.items()]
display(spark.createDataFrame(summary, "dataframe string, rows long, n_cols int, columns string"))

## 2. Transform and write to silver

Each dataset is cleaned or enriched, then written as **Parquet** to the `silver` container at `abfss://silver@advworksstorageacc.dfs.core.windows.net/AdventureWorks_<Name>`. Writes use `overwrite` mode, so re-running a cell replaces the folder instead of duplicating rows.

Write access comes from the Unity Catalog external location **`silver_ext`** (same `adventureworks_cred` credential as bronze).

The next cell imports PySpark functions as `f` (used as `f.col`, `f.split`, etc.).

In [0]:
from pyspark.sql import functions as f

### 2.1 Calendar

Adds `Month` and `Year` integer columns derived from `Date`, so downstream reports can group by period without recomputing them. This cell also defines the `silver` base path used by every later write.

In [0]:

silver = "abfss://silver@advworksstorageacc.dfs.core.windows.net"

df_calendar = (df_calendar
               .withColumn("Month", f.month(f.col("Date")))
               .withColumn("Year", f.year(f.col("Date"))))

(df_calendar.write.format("parquet")
    .mode("overwrite")
    .save(f"{silver}/AdventureWorks_Calendar"))

display(df_calendar.limit(5))

### 2.2 Write helper

`write_silver(df, name)` writes a dataframe as Parquet to `silver/AdventureWorks_<name>` in `overwrite` mode, so every dataset is saved the same way.

In [0]:
def write_silver(df, name):
    (df.write.format("parquet")
       .mode("overwrite")
       .save(f"{silver}/AdventureWorks_{name}"))

### 2.3 Products

* `ProductSKU`: keeps only the part before the first `-` (`HL-U509-R` becomes `HL`), which identifies the product family.
* `ProductName`: keeps only the first word (`Sport-100 Helmet, Red` becomes `Sport-100`).

The full descriptive name is still available in `ModelName`.

In [0]:
# Classic cleanup: keep SKU prefix (e.g. 'HL-U509-R' -> 'HL') and first word of product name
df_products = (df_products
               .withColumn("ProductSKU", f.split(f.col("ProductSKU"), "-")[0])
               .withColumn("ProductName", f.split(f.col("ProductName"), " ")[0]))
write_silver(df_products, "Products")
display(df_products.limit(5))

### 2.4 Sales

* **Union:** combines Sales 2015, 2016 and 2017 into one `df_sales` (`unionByName` matches columns by name, not position). Expected rows: 2,630 + 23,935 + 29,481 = 56,046.
* **`StockDate`:** converted from date to timestamp.
* **`OrderNumber`:** the `S` prefix becomes `T` (`SO45080` becomes `TO45080`).
* **`Multiply`:** new column = `OrderLineItem` × `OrderQuantity`.

Note: some `StockDate` values in the source data fall in 2001, before the order dates. That comes from the bronze files, not from this transformation.

In [0]:
df_sales = (df_sales_2015
            .unionByName(df_sales_2016)
            .unionByName(df_sales_2017)
            .withColumn("StockDate", f.to_timestamp(f.col("StockDate")))
            .withColumn("OrderNumber", f.regexp_replace(f.col("OrderNumber"), "S", "T"))
            .withColumn("Multiply", f.col("OrderLineItem") * f.col("OrderQuantity")))
write_silver(df_sales, "Sales")
display(df_sales.limit(5))

### 2.5 Lookup tables (no changes)

Product Categories, Product Subcategories, Returns and Territories are already clean, so they are written to silver as they are.

In [0]:
# Already clean - written to silver without changes
for name, df in {
    "Product_Categories": df_prod_cat,
    "Product_Subcategories": df_prod_subcat,
    "Returns": df_returns,
    "Territories": df_territories,
}.items():
    write_silver(df, name)

## 3. Verify silver outputs

Reads every silver Parquet folder back from Azure and shows its row count and columns. Each count should match its bronze source (Sales should equal the sum of the three yearly files).

In [0]:
names = ["Calendar", "Customers", "Product_Categories", "Product_Subcategories",
         "Products", "Returns", "Sales", "Territories"]
rows = []
for n in names:
    d = spark.read.parquet(f"{silver}/AdventureWorks_{n}")
    rows.append((n, d.count(), ", ".join(d.columns)))
display(spark.createDataFrame(rows, "table string, rows long, columns string"))

### Customers

Adds `fullName` by joining `Prefix`, `FirstName` and `LastName` with spaces (`concat_ws` skips null parts, so a missing prefix leaves no extra space), then writes the result to `silver/AdventureWorks_Customers`.

Note: the write below uses `append` mode, so each re-run adds all customers again. Switch to `overwrite` to keep it re-runnable.

In [0]:
df_customers = df_customers.withColumn('fullName', f.concat_ws(' ', f.col('Prefix'), f.col('FirstName'), f.col('LastName')))

In [0]:
(df_customers.write.format("parquet")
    .mode("append")
    .save(f"{silver}/AdventureWorks_Customers"))